## 5.1 Ingesting Data & Stripping Whitespace Quirks

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv('data/loan_approval_dataset.csv')

# Clean leading/trailing spaces from column headers
df.columns = df.columns.str.strip()

# Clean leading/trailing spaces from string text values inside the columns
# (checks both legacy 'object' dtype and pandas 3.0+ 'str' dtype)
for col in df.columns:
    if df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col]):
        df[col] = df[col].str.strip()

print("Dataset Ingested & Whitespaces Cleaned Successfully!")
print(f"Dataset Shape: {df.shape} (Rows, Columns)")
print("\nFirst 5 Records:")
print(df.head())

## 5.2 Statistical Summaries & Data Quality Audits

In [ ]:
print("Data Summary Statistics:")
print(df.describe())

print("\nMissing Values per Feature:")
print(df.isnull().sum())

print("\nTarget Class Distribution:")
print(df['loan_status'].value_counts(normalize=True) * 100)

## 5.3 Correlation Heatmap of Financial Features

In [ ]:
plt.figure(figsize=(10, 8))
numeric_cols = df.select_dtypes(include=[np.number]).drop(columns=['loan_id'])
sns.heatmap(numeric_cols.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Financial Features')
plt.show()

## 5.4 Split, Label Encode, Scale, and Train Baseline Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Drop redundant identifiers
X = df.drop(columns=['loan_id', 'loan_status'])
y = df['loan_status']

# Label encode categorical text columns
le = LabelEncoder()
cat_cols = ['education', 'self_employed']
for col in cat_cols:
    X[col] = le.fit_transform(X[col])

# Encode target values ('Approved' -> 0, 'Rejected' -> 1)
y = y.map({'Approved': 0, 'Rejected': 1})

# Train-Test Split (80% training, 20% validation testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit and apply StandardScaler to normalize continuous ranges
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train a baseline Random Forest Classifier
clf = RandomForestClassifier(random_state=42, n_estimators=100)
clf.fit(X_train_scaled, y_train)

# Evaluate
y_pred = clf.predict(X_test_scaled)
print(f"Baseline Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))